#NB_patient_360

In [0]:
import time

_run_start = time.time()
target_table = "adb_healthcare.gold.patient_360"
print("=" * 70)
print(f"[START] NB_patient_360 building '{target_table}'")

In [0]:
%sql
CREATE OR REPLACE TABLE adb_healthcare.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary (STTM silver to gold)'
AS
WITH lab_results_dedup AS (
  -- silver.lab_results contains every test once per load batch: keep the latest version of each test
  SELECT *
  FROM adb_healthcare.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY landing_timestamp DESC, update_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    count(*)                                 AS total_tests,
    count_if(result_status = 'Abnormal')     AS abnormal_test_count,
    count_if(result_status = 'Critical')     AS critical_test_count,
    max(test_date)                           AS last_test_date
  FROM lab_results_dedup
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                    AS patient_id,
  p.first_name                                                                 AS first_name,
  p.last_name                                                                  AS last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,  -- age in years as of today
  p.gender                                                                     AS gender,
  p.city                                                                       AS city,   -- patient's own city
  p.state                                                                      AS state,  -- patient's own state
  CAST(p.registered_hospital_id AS INT)                                        AS hospital_id,
  h.hospital_name                                                              AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                         AS insurance_provider_id,
  i.provider_name                                                              AS insurance_provider_name,
  CAST(coalesce(l.total_tests, 0) AS INT)                                      AS total_tests,
  CAST(coalesce(l.abnormal_test_count, 0) AS INT)                              AS abnormal_test_count,
  CAST(coalesce(l.critical_test_count, 0) AS INT)                              AS critical_test_count,
  l.last_test_date                                                             AS last_test_date
FROM adb_healthcare.silver.patients p
LEFT JOIN adb_healthcare.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_healthcare.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
records_count = spark.table(target_table).count()
print(f"[EXIT] Returning records_count = {records_count} to caller")
print(f"[END] NB_patient_360 finished for table '{target_table}' in {time.time() - _run_start:.1f}s")
print("=" * 70)
dbutils.notebook.exit(str(records_count))